# Srinagar-only IBM Granite LST fine-tuning (clean pilot)

This notebook deliberately preserves IBM's published Granite LST model configuration and checkpoint. It only changes the data locations and short pilot-training settings.

**Important data requirement:** IBM's model expects 7 input channels: six HLS bands (BLUE, GREEN, RED, NIR_NARROW, SWIR_1, SWIR_2) plus ERA5-Land 2 m air temperature in channel 7. Confirm that your seventh input band is ERA5-Land 2 m temperature, with the same units/scaling expected by the IBM configuration, before setting `BAND7_CONFIRMED = True`.

This is a pilot, not a guarantee of improvement. It uses six Srinagar acquisitions (if available), 224×224 patches, a spatial holdout for validation, a one-batch smoke test, then five training epochs.

In [ ]:
# CELL 1 — Mount Drive, imports, paths, and GPU check
from google.colab import drive
drive.mount("/content/drive")

import os, sys, copy, random, shutil, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
import yaml
import torch
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

WORKSPACE = Path("/content/drive/MyDrive/colab_workspace")
DATA_ROOT = WORKSPACE / "data" / "hls"
MODEL_DIR = WORKSPACE / "models_and_configs"

CITY = "srinagar_ind"
INPUT_DIR = DATA_ROOT / CITY / "hls-predict-ready"
TARGET_DIR = DATA_ROOT / CITY / "target-lst"

ALIGNED_TARGET_DIR = WORKSPACE / "data" / "srinagar_aligned" / CITY / "target-lst"
PATCH_ROOT = WORKSPACE / "data" / "granite_srinagar_pilot"
TRAIN_INPUT = PATCH_ROOT / "train" / "inputs"
TRAIN_TARGET = PATCH_ROOT / "train" / "targets"
VAL_INPUT = PATCH_ROOT / "val" / "inputs"
VAL_TARGET = PATCH_ROOT / "val" / "targets"

CHECKPOINT_DIR = MODEL_DIR / "srinagar_checkpoints"
RUN_DIR = WORKSPACE / "runs" / "srinagar_pilot"
LOG_DIR = WORKSPACE / "tb_logs"
FIT_CONFIG = MODEL_DIR / "srinagar_fit_config.yaml"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Do not launch the full training run on CPU.")

assert INPUT_DIR.is_dir(), f"Missing input directory: {INPUT_DIR}"
assert TARGET_DIR.is_dir(), f"Missing target directory: {TARGET_DIR}"

In [ ]:
# CELL 2 — Remove only outputs from the abandoned fine-tuning attempts
# This DOES NOT delete your original HLS data, target-LST data, base model checkpoint,
# original IBM config, or prediction outputs.

outputs_to_delete = [
    WORKSPACE / "data" / "hls_sanitized_srinagar_v1",
    WORKSPACE / "data" / "srinagar_aligned",
    WORKSPACE / "data" / "granite_srinagar_pilot",
    MODEL_DIR / "sanitization_reports_srinagar",
    MODEL_DIR / "srinagar_alignment_reports",
    MODEL_DIR / "checkpoints",                 # old custom fit outputs
    MODEL_DIR / "srinagar_checkpoints",        # this clean pilot's previous outputs, if any
    MODEL_DIR / "fit_config.yaml",             # old hand-written config
    MODEL_DIR / "srinagar_fit_config.yaml",
    WORKSPACE / "tb_logs" / "granite_lst_finetune",
    WORKSPACE / "tb_logs" / "srinagar_pilot",
    WORKSPACE / "runs" / "srinagar_pilot",
]

print("The following paths will be deleted if they exist:")
for p in outputs_to_delete:
    print(("FOUND  " if p.exists() else "absent ") + str(p))

confirmation = input("\nType DELETE-SRINAGAR-FINETUNE-OUTPUTS to proceed: ").strip()
if confirmation != "DELETE-SRINAGAR-FINETUNE-OUTPUTS":
    raise RuntimeError("Cleanup cancelled. No paths were deleted.")

for p in outputs_to_delete:
    if p.is_dir():
        shutil.rmtree(p)
    elif p.is_file():
        p.unlink()

print("Cleanup complete. Original source data and IBM base-model files were preserved.")

In [ ]:
# CELL 3 — Download the official IBM checkpoint and ORIGINAL config
# Do not hand-build or replace the model architecture. We will deep-copy IBM's config.

REPO_ID = "ibm-granite/granite-geospatial-land-surface-temperature"

CHECKPOINT = Path(hf_hub_download(
    repo_id=REPO_ID,
    filename="LST_model.ckpt",
    local_dir=str(MODEL_DIR)
))
ORIGINAL_CONFIG = Path(hf_hub_download(
    repo_id=REPO_ID,
    filename="config.yaml",
    local_dir=str(MODEL_DIR)
))

with open(ORIGINAL_CONFIG, "r") as f:
    original_cfg = yaml.safe_load(f)

print("Base checkpoint:", CHECKPOINT)
print("Original IBM config:", ORIGINAL_CONFIG)

data_original = original_cfg["data"]["init_args"]
print("Original dataset bands:", data_original.get("dataset_bands"))
print("Original output bands:", data_original.get("output_bands"))
print("Original means:", data_original.get("means"))
print("Original stds:", data_original.get("stds"))

assert len(data_original["means"]) == 7
assert len(data_original["stds"]) == 7
assert len(data_original["dataset_bands"]) == 7
assert len(data_original["output_bands"]) == 7

# IBM's model card specifies six HLS bands + ERA5-Land 2 m temperature.
# Set True ONLY after verifying the seventh band in your existing TIFFs is ERA5-Land T2m.
BAND7_CONFIRMED = False
if not BAND7_CONFIRMED:
    raise RuntimeError(
        "STOP: verify that band 7 is ERA5-Land 2 m temperature, then set "
        "BAND7_CONFIRMED = True and rerun this cell."
    )

In [ ]:
# CELL 4 — Match Srinagar inputs to LST targets and align targets to input grids

NODATA = -9999.0
input_files = sorted(INPUT_DIR.glob("*.tif"))
if not input_files:
    input_files = sorted(INPUT_DIR.glob("*.tiff"))

target_lookup = {p.name: p for p in TARGET_DIR.glob("*.tif")}
pairs = []

for img_path in input_files:
    target_path = target_lookup.get(img_path.stem + ".lst.tif")
    if target_path is None:
        continue
    with rasterio.open(img_path) as src:
        if src.count != 7:
            raise ValueError(f"{img_path.name}: expected 7 bands, found {src.count}")
    pairs.append((img_path, target_path))

print("Input TIFFs:", len(input_files))
print("Matched input/target pairs:", len(pairs))
if len(pairs) < 2:
    raise RuntimeError(
        "Need at least two matched Srinagar input/target pairs. "
        "Check filenames: each target should be <input_stem>.lst.tif"
    )

ALIGNED_TARGET_DIR.mkdir(parents=True, exist_ok=True)
alignment_rows = []

for img_path, target_path in tqdm(pairs, desc="Aligning LST targets"):
    out_path = ALIGNED_TARGET_DIR / (img_path.stem + ".lst.tif")

    with rasterio.open(img_path) as src, rasterio.open(target_path) as tgt:
        if src.crs is None or tgt.crs is None:
            raise ValueError(f"Missing CRS for {img_path.name} or {target_path.name}")

        target = tgt.read(1, masked=True).filled(np.nan).astype(np.float32)

        same_grid = (
            src.width == tgt.width and src.height == tgt.height
            and src.crs == tgt.crs and src.transform == tgt.transform
        )

        if same_grid:
            aligned = target
        else:
            aligned = np.full((src.height, src.width), np.nan, dtype=np.float32)
            reproject(
                source=target,
                destination=aligned,
                src_transform=tgt.transform,
                src_crs=tgt.crs,
                src_nodata=np.nan,
                dst_transform=src.transform,
                dst_crs=src.crs,
                dst_nodata=np.nan,
                resampling=Resampling.bilinear,
                init_dest_nodata=True,
            )

        valid = np.isfinite(aligned)
        if not valid.any():
            raise ValueError(f"No valid target overlap for {img_path.name}")

        aligned[~valid] = NODATA
        profile = src.profile.copy()
        profile.update(
            driver="GTiff", count=1, dtype="float32", nodata=NODATA,
            compress="deflate", predictor=3
        )
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(aligned, 1)
            dst.set_band_description(1, "Aligned_LST_target")

        alignment_rows.append({
            "input": img_path.name,
            "target": target_path.name,
            "aligned_target": out_path.name,
            "same_grid_before_alignment": same_grid,
            "valid_target_fraction": float(valid.mean()),
        })

alignment_report = pd.DataFrame(alignment_rows)
display(alignment_report.head())
print("Aligned targets:", len(alignment_report))

In [ ]:
# CELL 5 — Build a small Srinagar patch dataset (maximum six acquisitions)

PATCH_SIZE = 224
MAX_ACQUISITIONS = 6
MAX_TRAIN_PATCHES_PER_ACQUISITION = 24
MAX_VAL_PATCHES_PER_ACQUISITION = 8
MIN_INPUT_VALID_FRACTION = 0.90
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

means = np.asarray(data_original["means"], dtype=np.float32)
stds = np.asarray(data_original["stds"], dtype=np.float32)

for d in [TRAIN_INPUT, TRAIN_TARGET, VAL_INPUT, VAL_TARGET]:
    d.mkdir(parents=True, exist_ok=True)

# Spread the six pilot acquisitions across the available matched acquisitions
# rather than blindly taking the first six filenames.
if len(pairs) > MAX_ACQUISITIONS:
    idxs = np.linspace(0, len(pairs) - 1, MAX_ACQUISITIONS).round().astype(int)
    selected_pairs = [pairs[i] for i in sorted(set(idxs))]
else:
    selected_pairs = pairs

manifest = []

for img_path, _ in tqdm(selected_pairs, desc="Extracting patches"):
    aligned_target_path = ALIGNED_TARGET_DIR / (img_path.stem + ".lst.tif")

    with rasterio.open(img_path) as src, rasterio.open(aligned_target_path) as tgt:
        if src.count != 7 or tgt.count != 1:
            raise ValueError(f"Unexpected band count for {img_path.name}")
        if (src.width, src.height, src.crs, src.transform) != (
            tgt.width, tgt.height, tgt.crs, tgt.transform
        ):
            raise ValueError(f"Input and aligned target grids differ: {img_path.name}")

        h, w = src.height, src.width
        input_nodata = src.nodata
        target_nodata = tgt.nodata
        train_candidates, val_candidates = [], []

        for row in range(0, h - PATCH_SIZE + 1, PATCH_SIZE):
            for col in range(0, w - PATCH_SIZE + 1, PATCH_SIZE):
                win = Window(col, row, PATCH_SIZE, PATCH_SIZE)
                image = src.read(window=win).astype(np.float32)
                target = tgt.read(1, window=win).astype(np.float32)
                masks = src.read_masks(window=win)
                target_mask = tgt.read_masks(1, window=win)

                input_valid_per_band = np.isfinite(image) & (masks > 0)
                input_valid_per_band &= (image != NODATA)
                if input_nodata is not None:
                    input_valid_per_band &= (image != input_nodata)

                input_valid = np.all(input_valid_per_band, axis=0)
                target_valid = np.isfinite(target) & (target_mask > 0)
                target_valid &= (target != NODATA)
                if target_nodata is not None:
                    target_valid &= (target != target_nodata)

                if input_valid.mean() < MIN_INPUT_VALID_FRACTION:
                    continue
                # Keep target patches only when every pixel has a valid label.
                if not target_valid.all():
                    continue

                candidate = (row, col, image, target, input_valid_per_band)
                # Hold out the bottom-right spatial quadrant for validation.
                if row >= h // 2 and col >= w // 2:
                    val_candidates.append(candidate)
                else:
                    train_candidates.append(candidate)

        random.shuffle(train_candidates)
        random.shuffle(val_candidates)
        train_candidates = train_candidates[:MAX_TRAIN_PATCHES_PER_ACQUISITION]
        val_candidates = val_candidates[:MAX_VAL_PATCHES_PER_ACQUISITION]

        def write_candidate(candidate, split):
            row, col, image, target, band_valid = candidate

            # Replace invalid input pixels with IBM's per-band means.
            # After normalization these pixels become zero.
            for band in range(7):
                image[band][~band_valid[band]] = means[band]

            patch_id = f"{img_path.stem}_r{row}_c{col}"
            input_name = patch_id + ".inputs.tif"
            label_name = patch_id + ".lst.tif"

            if split == "train":
                image_path = TRAIN_INPUT / input_name
                label_path = TRAIN_TARGET / label_name
            else:
                image_path = VAL_INPUT / input_name
                label_path = VAL_TARGET / label_name

            image_profile = src.profile.copy()
            image_profile.update(
                driver="GTiff", height=PATCH_SIZE, width=PATCH_SIZE,
                count=7, dtype="float32", nodata=None, compress="deflate",
                predictor=3, transform=src.window_transform(
                    Window(col, row, PATCH_SIZE, PATCH_SIZE)
                )
            )
            image_profile.pop("blockxsize", None)
            image_profile.pop("blockysize", None)

            label_profile = image_profile.copy()
            label_profile.update(count=1, nodata=NODATA)

            with rasterio.open(image_path, "w", **image_profile) as dst:
                dst.write(image)
            with rasterio.open(label_path, "w", **label_profile) as dst:
                dst.write(target, 1)

            manifest.append({
                "patch_id": patch_id,
                "input": input_name,
                "target": label_name,
                "source_acquisition": img_path.name,
                "split": split,
                "row": row,
                "col": col,
            })

        for c in train_candidates:
            write_candidate(c, "train")
        for c in val_candidates:
            write_candidate(c, "val")

        print(
            f"{img_path.name}: "
            f"{len(train_candidates)} train, {len(val_candidates)} validation"
        )

manifest_df = pd.DataFrame(manifest)
if manifest_df.empty:
    raise RuntimeError("No patches were generated. Inspect input/target validity and patch sizes.")

manifest_df.to_csv(PATCH_ROOT / "manifest.csv", index=False)

def verify_pairs(image_dir, label_dir, image_suffix, label_suffix):
    image_files = {p.name for p in image_dir.glob("*.tif")}
    label_files = {p.name for p in label_dir.glob("*.tif")}
    image_ids = {n.replace(image_suffix, "") for n in image_files}
    label_ids = {n.replace(label_suffix, "") for n in label_files}
    assert image_ids == label_ids, (
        f"Input/label IDs differ. Images={len(image_ids)}, labels={len(label_ids)}"
    )
    assert image_ids, f"No files in {image_dir}"
    return len(image_ids)

n_train = verify_pairs(TRAIN_INPUT, TRAIN_TARGET, ".inputs.tif", ".lst.tif")
n_val = verify_pairs(VAL_INPUT, VAL_TARGET, ".inputs.tif", ".lst.tif")

print("\nPatch counts:", {"train": n_train, "validation": n_val})
print(manifest_df.groupby("split").size())
assert n_train > 0 and n_val > 0, "Both train and validation splits are required." 

In [ ]:
# CELL 6 — Create a pilot config by copying IBM's config, changing only data paths/run settings

cfg = copy.deepcopy(original_cfg)
data_cfg = cfg["data"]["init_args"]

data_cfg["train_data_root"] = str(TRAIN_INPUT)
data_cfg["train_label_data_root"] = str(TRAIN_TARGET)
data_cfg["val_data_root"] = str(VAL_INPUT)
data_cfg["val_label_data_root"] = str(VAL_TARGET)

data_cfg["img_grep"] = "*.inputs.tif"
data_cfg["label_grep"] = "*.lst.tif"
data_cfg["batch_size"] = 1
data_cfg["num_workers"] = 2
data_cfg["drop_last"] = False

# Keep IBM's original dataset_bands, output_bands, means, stds,
# transforms, model factory, backbone, decoder, auxiliary head, loss and ignore_index.
assert len(data_cfg["means"]) == 7 and len(data_cfg["stds"]) == 7
assert len(data_cfg["dataset_bands"]) == 7

trainer_cfg = cfg["trainer"]
trainer_cfg["accelerator"] = "gpu" if torch.cuda.is_available() else "cpu"
trainer_cfg["devices"] = 1
trainer_cfg["precision"] = "16-mixed" if torch.cuda.is_available() else "32-true"
trainer_cfg["max_epochs"] = 5
trainer_cfg["check_val_every_n_epoch"] = 1
trainer_cfg["num_sanity_val_steps"] = 1
trainer_cfg["log_every_n_steps"] = 1
trainer_cfg["default_root_dir"] = str(RUN_DIR)
trainer_cfg["enable_checkpointing"] = True

trainer_cfg["callbacks"] = [
    {
        "class_path": "lightning.pytorch.callbacks.ModelCheckpoint",
        "init_args": {
            "dirpath": str(CHECKPOINT_DIR),
            "filename": "srinagar-best-{epoch:02d}",
            "monitor": "val/loss",
            "mode": "min",
            "save_top_k": 1,
            "save_last": True,
        },
    },
    {
        "class_path": "lightning.pytorch.callbacks.EarlyStopping",
        "init_args": {
            "monitor": "val/loss",
            "mode": "min",
            "patience": 2,
        },
    },
]

# Keep the logger type from IBM's config, but send logs to a fresh pilot folder.
logger_cfg = trainer_cfg.get("logger")
if isinstance(logger_cfg, dict) and isinstance(logger_cfg.get("init_args"), dict):
    logger_cfg["init_args"]["save_dir"] = str(LOG_DIR)
    logger_cfg["init_args"]["name"] = "srinagar_pilot"

# Conservative fine-tuning LR for a small city-specific pilot.
if "optimizer" in cfg and isinstance(cfg["optimizer"], dict):
    opt_args = cfg["optimizer"].get("init_args", {})
    if "lr" in opt_args:
        opt_args["lr"] = 1.0e-5

if "lr_scheduler" in cfg and isinstance(cfg["lr_scheduler"], dict):
    sched_args = cfg["lr_scheduler"].get("init_args", {})
    if "monitor" in sched_args:
        sched_args["monitor"] = "val/loss"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

with open(FIT_CONFIG, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Pilot config saved:", FIT_CONFIG)
print("Training input:", data_cfg["train_data_root"])
print("Training labels:", data_cfg["train_label_data_root"])
print("Validation input:", data_cfg["val_data_root"])
print("Validation labels:", data_cfg["val_label_data_root"])
print("Model factory:", cfg["model"]["init_args"].get("model_factory"))
print("Backbone:", cfg["model"]["init_args"]["model_args"].get("backbone"))
print("Learning rate:", cfg.get("optimizer", {}).get("init_args", {}).get("lr"))

In [ ]:
# CELL 7 — One-batch smoke test (this is NOT the full training run)
# It checks config parsing, checkpoint loading, dataloader construction,
# and one train/validation step before spending time on all epochs.

import subprocess

assert torch.cuda.is_available(), "No GPU is available. Select a Colab GPU runtime first."
assert CHECKPOINT.exists(), f"Missing base checkpoint: {CHECKPOINT}"

cmd = [
    "terratorch", "fit",
    "-c", str(FIT_CONFIG),
    "--ckpt_path", str(CHECKPOINT),
    "--trainer.fast_dev_run", "true",
]

print("Running one-batch smoke test:")
print(" ".join(cmd))
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        f"Smoke test failed with exit code {result.returncode}. "
        "Do not launch full training; inspect the first traceback above."
    )

print("SMOKE TEST PASSED — safe to proceed to the five-epoch pilot.")

In [ ]:
# CELL 8 — Actual Srinagar pilot fine-tuning (five epochs maximum)
# Run ONLY after Cell 7 reports SMOKE TEST PASSED.

cmd = [
    "terratorch", "fit",
    "-c", str(FIT_CONFIG),
    "--ckpt_path", str(CHECKPOINT),
]

print("Starting Srinagar pilot fine-tuning...")
print("Base checkpoint:", CHECKPOINT)
print("Training patches:", len(list(TRAIN_INPUT.glob("*.inputs.tif"))))
print("Validation patches:", len(list(VAL_INPUT.glob("*.inputs.tif"))))
print("Output checkpoints:", CHECKPOINT_DIR)

result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(
        f"Training failed with exit code {result.returncode}. "
        "Share the complete output above; do not delete the run logs."
    )

print("Pilot training completed.")
print("Saved checkpoint files:")
for p in sorted(CHECKPOINT_DIR.glob("*.ckpt")):
    print(p.name, f"{p.stat().st_size / 1e6:.1f} MB")

## Before interpreting results

- Compare the fine-tuned checkpoint with the original IBM checkpoint on the **same untouched Srinagar test patches**. Do not use training patches for evaluation.
- The current validation split is a spatial holdout within selected Srinagar acquisitions. It does not establish generalisation to other cities or dates.
- Check the target LST units and nodata handling. Do not clip valid cold Srinagar temperatures merely because they are below 0 °C.
- Six acquisitions and a few hundred patches are only a smoke-test/pilot dataset. A lower training loss alone does not prove better real-world predictions.